### 這個是拿來加入模組的

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import yfinance as yf
import openpyxl
import zipfile
import os
from scipy.stats import norm
from scipy.optimize import brentq
import os
import glob
import platform

### 我本來想解釋，但算了，反正這個不跑下面跑不動

In [ ]:
def get_interest_df(file_path='interest.xls'):
    """
    讀取利率
    """
    df = pd.read_excel(file_path, header=2)
    df = df.iloc[:, [0, 13]]
    df = df.loc[2:].copy()
    df.rename(columns={df.columns[0]: '年月', df.columns[1]: 'interest'}, inplace=True)
    df.columns = df.columns.str.strip()
    
    def convert_roc_to_ad(x):
        s = str(x).zfill(5)
        ad_year = int(s[:3]) + 1911
        month = s[3:]
        return f"{ad_year}/{month}"
    
    df['年月'] = pd.to_datetime(df['年月'].apply(convert_roc_to_ad), format='%Y/%m')
    df['年月'] = df['年月'].dt.to_period('M')
    df['interest'] = pd.to_numeric(df['interest'], errors='coerce') * 0.01
    return df.reset_index(drop=True)

def get_settle_df(file_path='taifex.csv'):
    """
    讀期交所結算資料 (僅保留月結算合約)
    """
    df = pd.read_csv(file_path)
    df = df[~df['contract'].str.contains('W')]
    df['settledate'] = pd.to_datetime(df['settledate'])
    return df.sort_values('settledate')

def get_twii(start='2024-01-01', end='2024-12-31'):
    """
    抓台指
    """
    df = yf.download('^twii', start=start, end=end)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df
    
def generate_path_file(start_date, end_date, df_interest, settledate_df, twii_df, output_path='Path.csv'):
    """
    合成 Path
    """
    twii = twii_df.copy()
    if isinstance(twii, pd.DataFrame):
        if isinstance(twii.columns, pd.MultiIndex):
            twii.columns = twii.columns.get_level_values(0)
    twii.index = pd.to_datetime(twii.index)
    
    dates = pd.date_range(start=start_date, end=end_date, freq='B')
    data = []
    
    for date in dates:
        date_str = date.strftime('%Y/%#m/%#d')
        file_name = f"OptionsDaily_{date.strftime('%Y_%m_%d')}.csv"
        
        # rf
        current_period = date.to_period('M')
        interest_row = df_interest[df_interest['年月'] == current_period]
        rf = interest_row['interest'].values[0] if not interest_row.empty else np.nan
        
        # Maturity / Contract
        future_options = settledate_df[settledate_df['settledate'] > date]
        if not future_options.empty:
            future_settle = future_options.iloc[0]
            contract = future_settle['contract']
            maturity = (future_settle['settledate'] - date).days
        else:
            contract, maturity = "N/A", np.nan
            
        # S0
        try:
            if isinstance(twii, pd.DataFrame):
                s0 = twii.loc[:date, 'Close'].iloc[-1]
            else:
                s0 = twii.loc[:date].iloc[-1]
        except:
            s0 = np.nan
            
        data.append([date_str, file_name, s0, maturity, contract, rf])
    
    df_path = pd.DataFrame(data, columns=['Date', 'File', 'S0', 'Maturity', 'Contract', 'rf'])
    df_path.dropna(subset='S0', inplace=True)
    df_path.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"Path 檔案已產生：{output_path}")
    return df_path

def bs_price(S, K, T, r, sigma, option_type):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'Call':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def find_iv(price, S, K, T, r, option_type):
    if price <= 0 or T <= 0: return np.nan
    try:
        return brentq(lambda x: bs_price(S, K, T, r, x, option_type) - price, 1e-6, 5)
    except:
        return np.nan

def calculate_iv_batch(path_csv, zip_path, output_dir='IV_Results'):
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    
    df_path = pd.read_csv(path_csv)
    df_path['Date'] = pd.to_datetime(df_path['Date'])
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        namelist = z.namelist()
        # 取得 zip 內的資料夾前綴（如 "Option_2021/"）
        folders = set(n.rsplit('/', 1)[0] for n in namelist if '/' in n)
        
        for idx, row in df_path.iterrows():
            curr_date = row['Date']
            
            # 兩種可能的檔名
            candidates = [
                row['File'],                                              # OptionsDaily_2021_01_04.csv
                f"o{curr_date.strftime('%Y%m%d')}.csv",                   # o20210104.csv
            ]
            
            # 嘗試各種資料夾 + 檔名組合
            target_file = None
            for folder in folders:
                for name in candidates:
                    full_path = f"{folder}/{name}"
                    if full_path in namelist:
                        target_file = full_path
                        break
                if target_file:
                    break
            
            # 也試無資料夾的情況
            if not target_file:
                for name in candidates:
                    if name in namelist:
                        target_file = name
                        break
            
            if not target_file:
                print(f"跳過: {curr_date.strftime('%Y-%m-%d')} (找不到檔案)")
                continue
            
            print(f"處理中: {target_file}")
            
            chunks = []
            encoding_to_use = 'utf-8'
            
            try:
                with z.open(target_file) as f:
                    pd.read_csv(f, nrows=5, encoding='utf-8')
            except UnicodeDecodeError:
                encoding_to_use = 'cp950'
            
            with z.open(target_file) as f:
                reader = pd.read_csv(f, encoding=encoding_to_use, chunksize=50000)
                for chunk in reader:
                    chunk.columns = chunk.columns.str.strip().str.replace('*', '', regex=False)
                    chunk = chunk[~chunk['成交日期'].astype(str).str.contains('---', na=False)]
                    chunk['成交數量(B or S)'] = pd.to_numeric(chunk['成交數量(B or S)'], errors='coerce')
                    filtered_chunk = chunk[
                        (chunk['商品代號'].astype(str).str.strip() == 'TXO') &
                        (chunk['成交數量(B or S)'] > 30)
                    ].copy()
                    if not filtered_chunk.empty:
                        chunks.append(filtered_chunk)
            
            if not chunks: continue
            
            df = pd.concat(chunks, ignore_index=True)
            df['成交價格'] = pd.to_numeric(df['成交價格'], errors='coerce')
            df['履約價格'] = pd.to_numeric(df['履約價格'], errors='coerce')
            
            S, r, T = row['S0'], row['rf'], row['Maturity'] / 365
            
            df['IV'] = df.apply(lambda x: find_iv(
                x['成交價格'], S, x['履約價格'], T, r,
                'Call' if str(x['買賣權別']).strip() == 'C' else 'Put'
            ), axis=1)
            
            out_name = f"IV_{curr_date.strftime('%Y%m%d')}.csv"
            df.to_csv(os.path.join(output_dir, out_name), index=False, encoding='utf-8-sig')

def merge_iv_stats_to_path(iv_dir='IV_Results', path_csv='Path.csv'):
    """讀取 IV_Results，計算多種選擇權指標，合併回 Path.csv"""
    
    all_files = glob.glob(os.path.join(iv_dir, 'IV_*.csv'))
    if not all_files:
        print("找不到 IV 結果檔案")
        return
    
    df_path = pd.read_csv(path_csv)
    old_cols = [c for c in df_path.columns if c not in ['Date', 'File', 'S0', 'Maturity', 'Contract', 'rf']]
    df_path.drop(columns=old_cols, inplace=True, errors='ignore')
    df_path['Date'] = pd.to_datetime(df_path['Date'])
    s0_map = df_path.set_index('Date')['S0'].to_dict()
    
    dfs = []
    for f in all_files:
        df = pd.read_csv(f)
        df.columns = df.columns.str.strip()
        dfs.append(df)
    
    df_all = pd.concat(dfs, ignore_index=True)
    df_all['IV'] = pd.to_numeric(df_all['IV'], errors='coerce')
    df_all['成交日期'] = df_all['成交日期'].astype(str).str.strip()
    df_all['買賣權別'] = df_all['買賣權別'].astype(str).str.strip()
    df_all['履約價格'] = pd.to_numeric(df_all['履約價格'], errors='coerce')
    df_all['成交數量(B or S)'] = pd.to_numeric(df_all['成交數量(B or S)'], errors='coerce')
    df_all = df_all.dropna(subset=['IV'])
    df_all['date_dt'] = pd.to_datetime(df_all['成交日期'], format='%Y%m%d')
    
    results = []
    
    for date_str, group in df_all.groupby('成交日期'):
        date_dt = pd.to_datetime(date_str, format='%Y%m%d')
        s0 = s0_map.get(date_dt, np.nan)
        
        calls = group[group['買賣權別'] == 'C']
        puts  = group[group['買賣權別'] == 'P']
        
        call_vol = calls['成交數量(B or S)'].sum()
        put_vol  = puts['成交數量(B or S)'].sum()
        
        # Put/Call Ratio
        pc_ratio = put_vol / call_vol if call_vol > 0 else np.nan
        
        # Call/Put IV 平均 & 標準差
        call_iv_mean = calls['IV'].mean() if len(calls) > 0 else np.nan
        call_iv_std  = calls['IV'].std()  if len(calls) > 0 else np.nan
        put_iv_mean  = puts['IV'].mean()  if len(puts) > 0 else np.nan
        put_iv_std   = puts['IV'].std()   if len(puts) > 0 else np.nan
        
        # Volume-Weighted IV
        if call_vol > 0:
            call_vwiv = (calls['IV'] * calls['成交數量(B or S)']).sum() / call_vol
        else:
            call_vwiv = np.nan
        if put_vol > 0:
            put_vwiv = (puts['IV'] * puts['成交數量(B or S)']).sum() / put_vol
        else:
            put_vwiv = np.nan
        
        # ATM IV（最接近 S0 的履約價）
        atm_iv_call = atm_iv_put = np.nan
        if not np.isnan(s0):
            if len(calls) > 0:
                atm_call = calls.iloc[(calls['履約價格'] - s0).abs().argsort()[:1]]
                atm_iv_call = atm_call['IV'].values[0]
            if len(puts) > 0:
                atm_put = puts.iloc[(puts['履約價格'] - s0).abs().argsort()[:1]]
                atm_iv_put = atm_put['IV'].values[0]
        
        # IV Skew = OTM Put 平均 IV − ATM IV
        # OTM Put: 履約價 < S0
        iv_skew = np.nan
        if not np.isnan(s0) and not np.isnan(atm_iv_put):
            otm_puts = puts[puts['履約價格'] < s0]
            if len(otm_puts) > 0:
                iv_skew = otm_puts['IV'].mean() - atm_iv_put
        
        # IV Spread = Put IV mean − Call IV mean
        iv_spread = put_iv_mean - call_iv_mean if not (np.isnan(put_iv_mean) or np.isnan(call_iv_mean)) else np.nan
        
        # IV Range
        iv_range = group['IV'].max() - group['IV'].min()
        
        results.append({
            'Date': date_dt,
            'PC_Ratio': pc_ratio,
            'Call_IV_mean': call_iv_mean,
            'Call_IV_std': call_iv_std,
            'Put_IV_mean': put_iv_mean,
            'Put_IV_std': put_iv_std,
            'Call_VWIV': call_vwiv,
            'Put_VWIV': put_vwiv,
            'ATM_IV_Call': atm_iv_call,
            'ATM_IV_Put': atm_iv_put,
            'IV_Skew': iv_skew,
            'IV_Spread': iv_spread,
            'IV_Range': iv_range,
        })
    
    df_stats = pd.DataFrame(results)
    
    df_path = df_path.merge(df_stats, on='Date', how='left')
    
    if platform.system() == 'Windows':
        fmt = '%Y/%#m/%#d'
    else:
        fmt = '%Y/%-m/%-d'
    df_path['Date'] = df_path['Date'].dt.strftime(fmt)
    
    df_path.to_csv(path_csv, index=False, encoding='utf-8-sig')
    print(f"已寫入 {path_csv}，共 {len(df_stats.columns)-1} 個指標")
    return df_path

### 設定時間
你自己設定日期，你做 2024 年就設定 '2024-01-01', '2024-12-31'  
我設定 2021 是因為我只有2021的資料

In [ ]:
start, end = '2021-01-01', '2021-12-31'
df_interest = get_interest_df()
df_settle = get_settle_df()
twii = get_twii(start, end)
df_path_final = generate_path_file(start, end, df_interest, df_settle, twii)

### 有了 path 之後這個可以幫你算 iv，會新增一個資料夾，叫 IV_Result

In [ ]:
calculate_iv_batch('Path.csv', 'Option_2021[1].zip')

### 有 IV_Result 後，這個會自己算指標，再放回 Path 裡面
我還幫你多加了幾個指標，看不懂就複製上面很多 def 那邊，然後問 ai

In [ ]:
df_result = merge_iv_stats_to_path()

### 然後你就可以一邊打開你的漂亮 ppt，一邊想明天要怎麼報答我